<a target="_blank" href="https://colab.research.google.com/github/agensflow-ai/agensflow-langgraph/blob/main/notebooks/quickstart.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AgensFlow LangGraph — quickstart

This notebook boots the whole free-tier bundle **in one Python kernel** and
runs a real MAS end-to-end:

1. Start the AgensFlow policy server in-process (no separate `uvicorn`)
2. Issue a user API key
3. Import a converged starter policy — the substrate skips cold-start
4. Build the `parallel_critic_mas` graph
5. Run a real query against OpenRouter
6. Inspect the routing decisions the substrate made

**Prerequisites:** `OPENROUTER_API_KEY` in your environment. That's it — no
hosted server, no auth setup, no external DB.

The whole notebook runs top-to-bottom in ~2 minutes and costs ~$0.30 in
OpenRouter fees for the one demo query.

## 0. Colab setup (auto-skipped if running locally)

One-shot cell: installs the two packages from PyPI, clones the repo to get
the `examples/` directory the graph builder imports from, `cd`s into the
notebooks folder so the relative paths in later cells resolve, and prompts
for `OPENROUTER_API_KEY` if it isn't already in the environment.

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '-q',
         'agensflow-mcp', 'agensflow-langgraph',
         'asgi-lifespan', 'langchain-openai', 'python-dotenv'],
        check=True,
    )
    # Clone the repo to get examples/prompts + documents/starter_policies.
    if not os.path.isdir('agensflow-langgraph'):
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/agensflow-ai/agensflow-langgraph.git'],
            check=True,
        )
    os.chdir('agensflow-langgraph/notebooks')
    print(f'  ✓ Colab environment ready — cwd = {os.getcwd()}')

# --- OpenRouter API key setup ---
# Option 1 (preferred): a `.env` file with:
#     OPENROUTER_API_KEY=your_key_here
# Option 2: in-cell magic:
#     %env OPENROUTER_API_KEY=your_key_here
# Option 3: paste when prompted below.

from dotenv import load_dotenv
load_dotenv()
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

if not OPENROUTER_API_KEY:
    print('⚠️  OPENROUTER_API_KEY not found. Set it with %env, a .env file, '
          'or paste it below.')
    OPENROUTER_API_KEY = input('OPENROUTER_API_KEY: ').strip()

os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
print('✅ OPENROUTER_API_KEY loaded')

## 1. Boot the policy server in-process

We use `httpx.ASGITransport` to run the FastAPI app inside this Python
process — same trick our integration tests use. All `/langgraph/*` requests
go through the ASGI transport instead of real HTTP, but everything else
(bandits, storage, tenant isolation) works exactly like a real deployment.

In [ ]:
import os

# Force SQLite-in-memory for this notebook run so we don't need ./data/
os.environ.setdefault('AGF_DATABASE_URL', 'sqlite+aiosqlite:///:memory:')
os.environ.setdefault('AGF_ENV', 'test')
os.environ.setdefault('AGF_JWT_SECRET', 'notebook-not-for-production')

from httpx import ASGITransport, AsyncClient
from asgi_lifespan import LifespanManager

from agensflow_mcp.app import create_app
from agensflow_mcp.db.session import init_db, get_engine
from agensflow_mcp.db.models import Base

app = create_app()
await init_db()
engine = get_engine()
async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

lifespan_mgr = LifespanManager(app)
await lifespan_mgr.__aenter__()
server_client = AsyncClient(transport=ASGITransport(app=app), base_url='http://test')
print('  ✓ policy server booted in-process')

## 2. Issue an anonymous API key

The same endpoint a real deployment exposes — `POST /auth/anonymous`.

In [ ]:
resp = await server_client.post('/auth/anonymous')
api_key = resp.json()['api_key']
print(f'  api_key: {api_key[:20]}...')

## 3. Route the adapter's HTTP calls through our in-process server

Normally `agensflow-langgraph`'s client hits a real HTTPS server. Here we
monkey-patch it to use the in-process ASGI transport — one small class
override, then all the decorator's `/langgraph/*` calls flow through the
notebook's own kernel.

In [ ]:
from agensflow_langgraph import client as agf_client

class _NotebookClient(agf_client.AgensFlowClient):
    async def _a_post_model(self, path, payload, model_cls):
        r = await server_client.post(path, json=payload, headers=self._headers)
        self._raise_for_status(r)
        return model_cls.model_validate(r.json())

    async def _a_get_model(self, path, model_cls, params=None):
        r = await server_client.get(path, params=params, headers=self._headers)
        self._raise_for_status(r)
        return model_cls.model_validate(r.json())

agf_client._CACHE.clear()
agf_client.AgensFlowClient = _NotebookClient
os.environ['AGENSFLOW_SERVER_URL'] = 'http://test'
os.environ['AGENSFLOW_API_KEY'] = api_key
print('  ✓ adapter wired to in-process server')

## 4. Import the converged starter policy

`parallel_critic_v1.json` is the exported bandit state from 40 real runs of
our example MAS. Importing it gives the substrate warm priors on the arms —
no cold-start exploration needed for signatures whose names match ours.

In [ ]:
from pathlib import Path
from agensflow_langgraph import aimport_policy

# Path is relative to the repo root — the notebook lives in notebooks/
policy_path = Path('..') / 'examples' / 'starter_policies' / 'parallel_critic_v1.json'
assert policy_path.exists(), f'Starter policy not found at {policy_path.resolve()}'

result = await aimport_policy(policy_path)
print(f'  ✓ imported {result["signatures_merged"]} signatures / '
      f'{result["actions_merged"]} arms into the substrate')

## 5. Build the parallel_critic MAS graph

This is where the substrate earns its keep. We build a 6-node LangGraph MAS
where **every node is decorated with `@agensflow(pool={...})`**. Each pool
declares 2–3 candidate models; the substrate learns per-node which one wins.

```
  START → planner → memory → solver ─┬─→ critic  ─┐
                                     └─→ verifier ─┴─→ evaluator → END
```

Critic + verifier run in **parallel** — a shape the decorator handles
without special-casing.

### 5a. Import schemas + prompts + document corpus

These are the *content* (system prompts, Pydantic response schemas, the tiny
TCP/UDP/DNS corpus). They're not architecture — they're what the graph
reasons over, and they'd be different for every real use case.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))  # so `examples/` is importable

from examples.parallel_critic_mas.prompts import (
    PlannerOutput, MemoryOutput, SolverOutput,
    CriticOutput, VerifierOutput, EvaluatorOutput,
    EvidenceItem,
    PLANNER_SYS, MEMORY_SYS, SOLVER_SYS,
    CRITIC_SYS, VERIFIER_SYS, EVALUATOR_SYS,
    format_planner_input, format_memory_input, format_solver_input,
    format_critic_input, format_verifier_input, format_evaluator_input,
)
from examples.parallel_critic_mas.documents import render_corpus
print('  ✓ schemas + prompts imported')

### 5b. Declare the per-node model pools

One pool per node. Every model is a `ChatOpenAI` pointed at OpenRouter —
one API key unlocks Anthropic + OpenAI + Meta + Google + everyone else.
`.with_structured_output(schema)` returns a Runnable that parses to the
Pydantic schema, which is what the substrate wraps.

Node topology observations:
- `planner` and `solver` need capable reasoners → include a `deep` arm
- `memory` is retrieval-oriented → cheap models often suffice
- `evaluator` has a single arm (nothing to learn) — the substrate falls
  through instantly

In [ ]:
from langchain_openai import ChatOpenAI

def or_model(model_id: str, schema):
    """OpenRouter ChatOpenAI client with structured output pinned to `schema`."""
    return ChatOpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        model=model_id,
        temperature=0.0,
        max_retries=2,
    ).with_structured_output(schema, method='function_calling')

pools = {
    'planner': {
        'fast': or_model('openai/gpt-4o-mini',              PlannerOutput),
        'deep': or_model('anthropic/claude-sonnet-4',        PlannerOutput),
    },
    'memory': {
        'cheap': or_model('meta-llama/llama-3.3-70b-instruct', MemoryOutput),
        'solid': or_model('openai/gpt-4o-mini',                MemoryOutput),
    },
    'solver': {
        'cheap':    or_model('meta-llama/llama-3.3-70b-instruct', SolverOutput),
        'balanced': or_model('openai/gpt-4o',                     SolverOutput),
        'deep':     or_model('anthropic/claude-sonnet-4',         SolverOutput),
    },
    'critic':    {'cheap': or_model('openai/gpt-4o-mini', CriticOutput),
                  'solid': or_model('openai/gpt-4o',       CriticOutput)},
    'verifier':  {'cheap': or_model('openai/gpt-4o-mini', VerifierOutput),
                  'solid': or_model('openai/gpt-4o',       VerifierOutput)},
    'evaluator': {'default': or_model('openai/gpt-4o-mini', EvaluatorOutput)},
}
for node, arms in pools.items():
    print(f'  {node:10s} — {len(arms)} arm(s): {list(arms)}')

### 5c. Decorate the 6 nodes with `@agensflow`

**This is the whole integration.** Each async node function gets
`@agensflow(pool=pools['name'])` prepended. The decorator:

1. Derives a signature from the node identity + graph context
2. Asks the substrate which arm to use (UCB1 over the pool keys)
3. Injects the chosen `model` into the function body
4. Captures cost + tokens + latency on the returned message
5. POSTs the outcome to `/langgraph/decision/execute` so the bandit updates

Everything after the `@agensflow` line is your ordinary LangGraph node.
The decorator is the only AgensFlow-specific code you write.

In [ ]:
import operator
from typing import Annotated, Any, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from agensflow_langgraph import agensflow

class MASState(TypedDict, total=False):
    user_task: str
    goal: str; subproblem: str
    evidence: list[dict]
    draft_answer: str; solver_reasoning: str
    reasoning_score: float; reasoning_issues: list[str]
    verifier_verdict: str; ungrounded_claims: list[str]
    final_answer: str; evaluator_reasoning: str
    messages: Annotated[list, add_messages]
    # critic + verifier fan-in write `trace` concurrently → concat reducer
    trace: Annotated[list, operator.add]

def _trace(node, model):
    cfg = getattr(model, 'config', None) or {}
    return {'node': node, 'action': (cfg.get('metadata') or {}).get('agensflow_action', '<fell-open>')}

@agensflow(pool=pools['planner'])
async def planner(state, model, config=None):
    r = await model.ainvoke([('system', PLANNER_SYS),
                             ('human', format_planner_input(state['user_task']))])
    return {'goal': r.goal, 'subproblem': r.subproblem, 'trace': [_trace('planner', model)]}

@agensflow(pool=pools['memory'])
async def memory(state, model, config=None):
    r = await model.ainvoke([('system', MEMORY_SYS.format(corpus=render_corpus())),
                             ('human', format_memory_input(state['subproblem']))])
    return {'evidence': [e.model_dump() for e in r.evidence], 'trace': [_trace('memory', model)]}

@agensflow(pool=pools['solver'])
async def solver(state, model, config=None):
    ev = [EvidenceItem(**e) for e in state.get('evidence', [])]
    r = await model.ainvoke([('system', SOLVER_SYS),
                             ('human', format_solver_input(state['subproblem'], ev))])
    return {'draft_answer': r.draft_answer, 'solver_reasoning': r.reasoning,
            'trace': [_trace('solver', model)]}

@agensflow(pool=pools['critic'])
async def critic(state, model, config=None):
    r = await model.ainvoke([('system', CRITIC_SYS),
                             ('human', format_critic_input(state['subproblem'],
                                                           state['draft_answer'],
                                                           state['solver_reasoning']))])
    return {'reasoning_score': r.reasoning_score, 'reasoning_issues': list(r.reasoning_issues),
            'trace': [_trace('critic', model)]}

@agensflow(pool=pools['verifier'])
async def verifier(state, model, config=None):
    ev = [EvidenceItem(**e) for e in state.get('evidence', [])]
    r = await model.ainvoke([('system', VERIFIER_SYS),
                             ('human', format_verifier_input(state['subproblem'],
                                                             state['draft_answer'], ev))])
    return {'verifier_verdict': r.verdict, 'ungrounded_claims': list(r.ungrounded_claims),
            'trace': [_trace('verifier', model)]}

@agensflow(pool=pools['evaluator'])
async def evaluator(state, model, config=None):
    r = await model.ainvoke([('system', EVALUATOR_SYS),
                             ('human', format_evaluator_input(state['goal'], state['draft_answer'],
                                                              state.get('reasoning_score', 0.0),
                                                              state.get('verifier_verdict', 'unknown')))])
    return {'final_answer': r.final_answer, 'evaluator_reasoning': r.merged_reasoning,
            'trace': [_trace('evaluator', model)]}

print('  ✓ 6 decorated nodes defined')

### 5d. Wire the StateGraph

Nothing AgensFlow-specific here — just plain LangGraph. The `add_edge` fan-out
(`solver → critic` AND `solver → verifier`) makes LangGraph execute both
concurrently. The fan-in (`critic → evaluator` AND `verifier → evaluator`)
makes it wait for both before running `evaluator`.

In [ ]:
graph = StateGraph(MASState)
graph.add_node('planner',   planner)
graph.add_node('memory',    memory)
graph.add_node('solver',    solver)
graph.add_node('critic',    critic)
graph.add_node('verifier',  verifier)
graph.add_node('evaluator', evaluator)

graph.add_edge(START, 'planner')
graph.add_edge('planner', 'memory')
graph.add_edge('memory',  'solver')
# Fan-out: critic + verifier both consume solver's output — run in parallel
graph.add_edge('solver',   'critic')
graph.add_edge('solver',   'verifier')
# Fan-in: BOTH must complete before evaluator runs
graph.add_edge('critic',   'evaluator')
graph.add_edge('verifier', 'evaluator')
graph.add_edge('evaluator', END)

compiled = graph.compile(checkpointer=InMemorySaver())
print('  ✓ graph compiled')
print(f'  nodes: {list(compiled.get_graph().nodes)}')

## 6. Run one real query end-to-end

The substrate picks per-node actions based on the imported priors + UCB1
exploration bonus. All 6 nodes route through the in-process server.

In [ ]:
user_task = (
    'What is the difference between TCP and UDP, and when should each be used? '
    'Answer using only the provided documents.'
)

result = await compiled.ainvoke(
    {'user_task': user_task, 'trace': []},
    config={'configurable': {'thread_id': 'notebook_demo_1'}},
)

print('  Final answer:')
print(f'    {result["final_answer"]}')
print()
print('  Substrate routed the graph as:')
for step in result['trace']:
    print(f'    {step["node"]:<10} → {step["action"]}')

## 7. Inspect the routing decisions server-side

Every routing decision (one per node × one per graph invocation) is
persisted to the server. Fetch them and see per-node cost, tokens, latency,
and the arm the substrate chose. Status is `executed` — cost/tokens
captured, awaiting a reward (which Section 8 will provide via the judge).

In [ ]:
# The run produced 6 decisions (one per node). Fetch them from the server —
# status will be 'executed' (cost + tokens captured, but no reward yet). 
# Section 8 below will fetch a real judge quality and submit it as the reward.

resp = await server_client.get(
    '/langgraph/decisions?limit=10',
    headers={'Authorization': f'Bearer {api_key}'},
)
for d in resp.json()['decisions']:
    print(f'  {d["signature"]:<10} action={d["action"]:<10} '
          f'tokens=in{d["tokens_input"] or 0}/out{d["tokens_output"] or 0}  '
          f'lat={d["latency_s"]:.1f}s  status={d["status"]}')

## 8. Where the value shows up

So far: your graph runs, the substrate routes each node, and cost + tokens
get captured server-side. That's the *plumbing*. The **value** — for a
product team evaluating this — is the story around it:

1. **How good was the answer?** — score it with the free-tier 3-panel judge
2. **Why did the substrate pick these arms?** — audit one decision end-to-end
3. **Does it actually learn?** — run a few more queries, watch the policy move

### 8a. Score the final answer with the 3-judge panel

The free tier ships a cross-family judge panel: 3 different-family models
(e.g. GPT, Claude, Llama) each score the answer on four axes
(*correctness, completeness, precision, robustness*), and the panel returns
the mean. The panel needs a **baseline** to anchor the scale — the rubric is
relative, not absolute. We generate a cheap single-model baseline first, then
compare the MAS's answer against it.

In [ ]:
from langchain_openai import ChatOpenAI
from agensflow_langgraph.judge_panel import relative_quality

# --- Cheap baseline: single-shot answer from gpt-4o-mini on the same task ---
baseline_llm = ChatOpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY'],
    model='openai/gpt-4o-mini',
    temperature=0.0,
)
baseline_sys = (
    'Answer concisely using only the provided context. If the context is empty, '
    'answer from general knowledge in one paragraph.'
)
baseline_msg = await baseline_llm.ainvoke([
    ('system', baseline_sys),
    ('human', user_task),
])
baseline_answer = baseline_msg.content
print(f'  ✓ baseline generated ({len(baseline_answer)} chars)')

# --- Panel: 3 cross-family judges score the MAS answer vs the baseline ---
PANEL_MODELS = (
    'openai/gpt-4o-mini',
    'anthropic/claude-haiku-4.5',
    'meta-llama/llama-3.3-70b-instruct',
)
quality, per_axis = await relative_quality(
    task=user_task,
    candidate=result['final_answer'],
    baseline=baseline_answer,
    openrouter_key=os.environ['OPENROUTER_API_KEY'],
    models=PANEL_MODELS,
)

print(f'\n  Panel quality (composed):  {quality:.3f}')
print('  Per-axis scores:')
for axis, score in per_axis.items():
    print(f'    {axis:14s} {score:.3f}')
print(f'\n  Panel: {", ".join(PANEL_MODELS)}')

### 8b. Audit one routing decision

Every decision the substrate makes is persisted with the full context. For
any given `decision_id`, you can retrieve: the signature (task-shape hash),
which arm was chosen, cost + tokens + latency, the quality it earned, and the
**current policy state** for that signature (visits + reward_mean per arm).
The last piece is what makes this auditable — you can always answer *"why
was this the reasonable choice given what the substrate knew?"*

In [ ]:
from agensflow_langgraph import arecord_reward
from agensflow_langgraph.client import get_client

# Submit the panel quality from 8a as the real reward.
await arecord_reward(quality=quality, thread_id='notebook_demo_1')
print(f'  ✓ submitted quality={quality:.3f} as reward for thread notebook_demo_1\n')

# Fetch the run's decisions and pick the solver's — the most interesting node
# (3 arms, so the substrate had a real choice to make).
resp = await server_client.get(
    '/langgraph/decisions?limit=20',
    headers={'Authorization': f'Bearer {api_key}'},
)
decisions = resp.json()['decisions']
solver_dec = next(d for d in decisions if d['signature'] == 'solver')

print('  === Decision audit — solver node ===')
print(f'    decision_id:   {solver_dec["decision_id"]}')
print(f'    signature:     {solver_dec["signature"]}')
print(f'    chose arm:     {solver_dec["action"]}')
print(f'    status:        {solver_dec["status"]}')
print(f'    cost:          ${solver_dec["cost_usd"] or 0:.4f}')
print(f'    tokens:        in={solver_dec["tokens_input"] or 0}  out={solver_dec["tokens_output"] or 0}')
print(f'    latency:       {solver_dec["latency_s"] or 0:.2f}s')
if solver_dec['quality'] is not None:
    print(f'    quality:       {solver_dec["quality"]:.3f}')
if solver_dec.get('reward_value') is not None:
    print(f'    reward_value:  {solver_dec["reward_value"]:.4f}')

# What did the substrate KNOW about each solver arm at time of choice?
# aexport_policy returns the WHOLE bandit state; slice the solver signature.
policy_resp = await get_client().a_export_policy()
solver_arms = policy_resp.policy.get('solver', {})

print('\n  === Solver arm stats (post-reward) ===')
print(f'    {"arm":<10} {"visits":>7} {"reward_mean":>12} {"reward_m2":>11}')
for arm, stats in sorted(solver_arms.items()):
    marker = '  ← chosen' if arm == solver_dec['action'] else ''
    print(f'    {arm:<10} {stats.get("visits", 0):>7} {stats.get("reward_mean", 0):>12.4f} '
          f'{stats.get("reward_m2", 0):>11.4f}{marker}')

### 8c. Does it actually learn? — 2 more queries + policy diff

One query barely nudges a bandit. To see the flywheel, we run 2 more queries
against the same graph, judge each, and compare the substrate's policy
*before* and *after*. What you should see: some arms' `reward_mean` climbs,
some drop, `visits` counts increment for every arm the substrate tried.

For a real production deployment, this same flywheel runs continuously —
every graph invocation is a bandit update, indefinitely.

In [ ]:
# --- Snapshot the current policy BEFORE we run more queries ---
policy_before = (await get_client().a_export_policy()).policy
print(f'  policy_before: {sum(len(v) for v in policy_before.values())} arms across '
      f'{len(policy_before)} signatures\n')

# --- Two more queries of varying task difficulty ---
more_tasks = [
    ('notebook_demo_2', 'What are the three main phases of the TLS 1.3 handshake?'),
    ('notebook_demo_3', 'How does HTTP/2 differ from HTTP/1.1, and what problem does it solve?'),
]

for thread_id, task in more_tasks:
    r = await compiled.ainvoke(
        {'user_task': task, 'trace': []},
        config={'configurable': {'thread_id': thread_id}},
    )
    # Generate a baseline and judge — same as 8a but inline
    base = (await baseline_llm.ainvoke([('system', 'Answer concisely.'), ('human', task)])).content
    q, _ = await relative_quality(
        task=task, candidate=r['final_answer'], baseline=base,
        openrouter_key=os.environ['OPENROUTER_API_KEY'], models=PANEL_MODELS,
    )
    await arecord_reward(quality=q, thread_id=thread_id)
    print(f'  {thread_id}: quality={q:.3f}  answer_len={len(r["final_answer"])}')

# --- Snapshot AFTER, then diff ---
policy_after = (await get_client().a_export_policy()).policy

print('\n  === Policy delta: what the substrate learned ===')
print(f'    {"signature":<12} {"arm":<12} {"visits Δ":>10} {"reward_mean Δ":>15}')
for sig in sorted(set(policy_before) | set(policy_after)):
    b_arms = policy_before.get(sig, {})
    a_arms = policy_after.get(sig, {})
    for arm in sorted(set(b_arms) | set(a_arms)):
        vb = b_arms.get(arm, {}).get('visits', 0)
        va = a_arms.get(arm, {}).get('visits', 0)
        mb = b_arms.get(arm, {}).get('reward_mean', 0.0)
        ma = a_arms.get(arm, {}).get('reward_mean', 0.0)
        dv = va - vb
        dm = ma - mb
        if dv > 0 or abs(dm) > 1e-4:
            arrow = '↑' if dm > 0 else ('↓' if dm < 0 else '·')
            print(f'    {sig:<12} {arm:<12} {dv:>+10} {dm:>+15.4f} {arrow}')

## 9. Cleanup

Close the in-process server.

In [ ]:
await server_client.aclose()
await lifespan_mgr.__aexit__(None, None, None)
print('  ✓ done')

## What you just saw

In this one notebook you:

- Booted a full AgensFlow policy server (bandits, tenant isolation, storage)
  inside the Python kernel
- Issued a real API key
- Warm-started the substrate from a converged 40-run policy
- Built a real MAS with parallel critic + verifier
- Ran one query end-to-end through OpenRouter with real token counting
- Verified the substrate recorded every routing decision with cost/latency

**That's the free-tier bundle.** For production, swap in a real `uvicorn`
server + Postgres, keep the exact same client + graph code — the ASGI
transport in cell 1 is the only thing that changes.

**Next:** see [`docs/integration.md`](../docs/integration.md) for the
advanced patterns (streaming, checkpointer, redaction, custom judges) and
[`examples/`](../examples/) for two more topology variations.